# 03 — Diffusion training

The model estimates the score of a variance-preserving SDE,

$$
dx = -\tfrac12 \beta(t) x\, dt + \sqrt{\beta(t)}\, dW.
$$

Training minimizes the weighted denoising score-matching loss
$\|\sigma_t s_\theta(x_t, t) + z\|^2$.
This cell uses the demo configuration so it can finish on CPU.


In [ ]:
from src.config.settings import demo_model_config, load_config, project_root
from src.data.loader import load_tabular
from src.services.experiments import ExperimentStore
from src.services.pipeline import prepare_training_data, profile_dataset, train_model

config = load_config()
frame = load_tabular(project_root() / config.data.sample_path).head(config.demo.max_rows)
profile = profile_dataset(frame, {
    'id_uniqueness_threshold': config.data.id_uniqueness_threshold,
    'high_cardinality_threshold': config.data.high_cardinality_threshold,
    'missing_warning_threshold': config.data.missing_warning_threshold,
    'duplicate_warning_threshold': config.data.duplicate_warning_threshold,
    'imbalance_warning_threshold': config.data.imbalance_warning_threshold,
})
prepared = prepare_training_data(
    frame, profile['roles'], config, seed=config.project.seed,
    holdout_ratio=0.15, validation_ratio=0.15,
    stratify_column=profile['suggested_target'], min_rows=32,
)
model_config = demo_model_config(config)
model_config['device'] = 'cpu'
record = ExperimentStore(project_root()).create({
    'dataset_name': 'sample_customer_data.csv',
    'rows': len(frame), 'columns': frame.shape[1], 'seed': config.project.seed,
    'model_config': model_config, 'demo_mode': True,
})
summary = train_model(
    prepared['arrays']['train'], prepared['arrays']['validation'], model_config,
    root=project_root(), experiment_id=record['experiment_id'],
    preprocessor=prepared['preprocessor'],
)
summary['best_val_loss'], summary['history']['train_loss']
